In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install open-clip-torch open3d opencv-python-headless -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.8 MB/s eta 0:00:00


In [3]:
import sys

PHASE0_DIR = '/content/drive/MyDrive/2026研究/ULIP_PointLLM/DentalPatchAligned3D/Phase0_Data'
if PHASE0_DIR not in sys.path:
    sys.path.insert(0, PHASE0_DIR)

In [4]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


## Step 1 — CLIP ランキング

In [5]:
import config as cfg

# パラメータ調整はここで行う
N_PATCHES    = cfg.N_PATCHES      # 1症例あたりのパッチ数
N_SAMPLE     = cfg.N_POINTS_FULL  # メッシュからのサンプリング点数
PATCH_RADIUS = cfg.PATCH_RADIUS   # ハイライト半径
TOP_K        = cfg.CLIP_TOP_K     # 提示する上位候補数
SAVE_RENDERS = True               # 8方位レンダリング画像を保存するか

print(f'PLY dir   : {cfg.PLY_DIR}')
print(f'Output    : {cfg.CANDIDATES_CLIP_JSON}')
print(f'Renders   : {cfg.RENDERS_DIR}/clip/')

PLY dir   : /content/drive/MyDrive/2026研究/Dataset/6167726/STLs/PLYs
Output    : /content/drive/MyDrive/2026研究/ULIP_PointLLM/data/DentalPatchData/dental_vocab/candidates_clip.json
Renders   : /content/drive/MyDrive/2026研究/ULIP_PointLLM/data/DentalPatchData/renders/clip/


In [ ]:
from tools.clip_text_ranker import main as clip_main

clip_main(
    n_sample     = N_SAMPLE,
    n_patches    = N_PATCHES,
    patch_radius = PATCH_RADIUS,
    top_k        = TOP_K,
    device       = device,
    save_renders = SAVE_RENDERS,
)

  [CLIP] Loading ViT-bigG-14 (laion2b_s39b_b160k) on cuda ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

  [CLIP] Model loaded.
Found 20 PLY files

[Pat 10na] Loading: Pat 10na_sample.ply
[Open3D WARNING] geometry::TriangleMesh appears to be a geometry::PointCloud (only contains vertices, but no triangles).
[Open3D WARNING] geometry::TriangleMesh appears to be a geometry::PointCloud (only contains vertices, but no triangles).
  Patch 000/31  LPS: L-A-S → "left sigmoid notch" (0.405)
  Patch 001/31  LPS: R-P-S → "right sigmoid notch" (0.376)
  Patch 002/31  LPS: R-A-I → "right mental tubercle" (0.359)
  Patch 003/31  LPS: L-A-I → "left mental tubercle" (0.379)
  Patch 004/31  LPS: R-P-I → "right mandibular angle" (0.368)
  Patch 005/31  LPS: R-A-S → "right sigmoid notch" (0.370)
  Patch 006/31  LPS: L-A-I → "left mental tubercle" (0.375)
  Patch 007/31  LPS: L-P-I → "left mandibular foramen" (0.376)
  Patch 008/31  LPS: R-P-S → "right sigmoid notch" (0.384)
  Patch 009/31  LPS: R-P-I → "right mandibular angle" (0.374)
  Patch 010/31  LPS: L-P-S 

## Step 3 — フォーマット変換 (人手確認 Step 2 完了後に実行)

In [ ]:
if not cfg.VERIFIED_CLIP_JSON.exists():
    raise FileNotFoundError(
        f'verified_labels_clip.json が見つかりません: {cfg.VERIFIED_CLIP_JSON}\n'
        'ローカルで Step 2 (人手確認) を完了してから再実行してください。'
    )

In [ ]:
from tools.build_patchalign_dataset import build_dataset

build_dataset(
    verified_path = cfg.VERIFIED_CLIP_JSON,
    ply_dir       = cfg.PLY_DIR,
    dataset_root  = cfg.DATASET_ROOT,
    split_dir     = cfg.SPLIT_DIR,
    n_final       = cfg.N_POINTS_FINAL,
    patch_radius  = cfg.PATCH_RADIUS,
    n_sample      = cfg.N_POINTS_FULL,
    val_ratio     = 0.2,
)

In [ ]:
from tools.build_anatomy_textbank import build_textbank

build_textbank(
    verified_path = cfg.VERIFIED_CLIP_JSON,
    vocab_dir     = cfg.VOCAB_DIR,
    device        = device,
)